# House Price Prediction Pipeline 🏠

An end-to-end machine learning pipeline built to predict real estate prices using robust statistical techniques and feature selection.

## 🚀 Key Features
* **Data Leakage Prevention:** Feature scaling (`StandardScaler`) is strictly applied *after* splitting data to protect test set integrity.
* **Automated Feature Selection:** Utilizes `LassoCV` with 5-fold cross-validation to algorithmically eliminate redundant or noisy features.
* **Unbiased Estimation:** Trains a standard `LinearRegression` model exclusively on Lasso's selected feature mask to remove regularization bias and maximize predictive accuracy.

## 🛠️ Tech Stack
* Python
* Pandas & NumPy
* Scikit-Learn
* Matplotlib / Seaborn

In [ ]:
import numpy as np
import pandas as pd 
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
data = pd.read_csv('train (1).csv')
print(data.shape)

In [ ]:
data.head()

In [ ]:
features_with_na = [f for f in data.columns if data[f].isnull().sum() > 0 ]
for feature in features_with_na :
    print(feature , np.round(data[feature].isnull().mean() * 100 , 4)  , " % missing value ")

In [ ]:
for feature in features_with_na :
    d = data.copy()
    d[feature] = np.where(d[feature].isnull() , 1 ,0 )
    d.groupby(feature)['SalePrice'].median().plot.bar()
    plt.title(feature)
    plt.show()

In [ ]:
numerical_features = [feature for feature in data.columns if not isinstance(d[feature].dropna().iloc[0] , str)]

data[numerical_features].head()

In [ ]:
year_features = [feature for feature in numerical_features if 'Yr' in feature or 'Year' in feature]
year_features

In [ ]:
for feature in year_features :
    if feature != 'YrSold' :
        d = data.copy()
        d[feature] = d['YrSold'] - d[feature]
        plt.scatter(d[feature] , d['SalePrice'])
        plt.xlabel(feature)
        plt.ylabel('SalePrice')
        plt.show()

In [ ]:
discret_features = [feature for feature in numerical_features if len(data[feature].unique()) <25 and feature not in year_features]
print(" discret_features count : {}".format(len(discret_features)))

In [ ]:
discret_features

In [ ]:
continuous_features = [feature for feature in numerical_features if feature not in discret_features + year_features + ['Id']]
print("continuous features count is : {} ".format(len(continuous_features)))

In [ ]:
for feature in continuous_features :
    d = data.copy()
    d[feature].hist(bins = 25)
    plt.xlabel(feature)
    plt.ylabel('count')
    plt.title(feature)
    plt.show()

In [ ]:
for feature in continuous_features :
    d = data.copy()
    d['SalePrice'] = np.log(d['SalePrice'])
    if 0 in d[feature].unique():
        pass
    else :
        d[feature]= np.log(d[feature])
        plt.scatter(d[feature] ,  d['SalePrice'] ) 
        plt.show()

In [ ]:
for feature in continuous_features :
    d = data.copy()
    d['SalePrice'] = np.log(d['SalePrice'])
    if 0 in d[feature].unique():
        pass
    else :
        d[feature]= np.log(d[feature])
        d.boxplot(column = feature)
        plt.ylabel(feature)
        plt.title(feature)
        plt.show()
        

In [ ]:
for feature in continuous_features :
    q1 = data[feature].quantile(0.25)
    q3 = data[feature].quantile(0.75)
    iqr = q3-q1
    lower_bound = q1 - (1.5 * iqr)
    upper_bound = q3 + (1.5 * iqr)
    data = data[(data[feature] >= lower_bound) & (data[feature] <= upper_bound)]

In [ ]:
for feature in continuous_features :
    d = data.copy()
    d['SalePrice'] = np.log(d['SalePrice'])
    if 0 in d[feature].unique():
        pass
    else :
        d[feature]= np.log(d[feature])
        d.boxplot(column = feature)
        plt.ylabel(feature)
        plt.title(feature)
        plt.show()

In [ ]:
categorical_features = [
    feature for feature in data.columns 
    if isinstance(data[feature].dropna().iloc[0], str)
]

In [ ]:
for feature in categorical_features:
    # Notice the 'p' added back into groupby
    data.groupby(feature)['SalePrice'].median().plot.bar()
    plt.xlabel(feature)
    plt.ylabel('SalePrice')
    plt.title(feature)
    plt.show()

In [ ]:
data['HouseAge'] = data['YrSold'] - data['YearBuilt']

In [ ]:
data.head()

In [ ]:
categorical_features_with_na = [feature for feature in categorical_features if data[feature].isnull().sum() > 0 ]
for feature in categorical_features_with_na :
    data[feature] = data[feature].fillna('Missing')
    

In [ ]:
data.head()

In [ ]:
numerical_featrues_with_na = [feature for feature in numerical_features if data[feature].isnull().sum() > 0 ]
for feature in numerical_featrues_with_na :
    data[feature] = data[feature].fillna(data[feature].dropna().mean())

In [ ]:
data['SalePrice'] = np.log(data['SalePrice'])
for feature in continuous_features :
    if 0 in data[feature].unique():
        pass
    else :
        data[feature]= np.log(data[feature])
        

In [ ]:
data.head()

In [ ]:
 # Calculate age for all year features relative to the year sold
for feature in ['YearBuilt', 'YearRemodAdd', 'GarageYrBlt']:
    data[feature] = data['YrSold'] - data[feature]
    
# After calculating the ages, we no longer need the raw 'YrSold' column
data = data.drop(['YrSold'], axis=1)

data[['YearBuilt', 'YearRemodAdd', 'GarageYrBlt']].head()

In [ ]:
for feature in categorical_features :
    ordered_labels = data.groupby(feature)['SalePrice'].mean().sort_values().index

    labels_ordering = {k : i for i,k in enumerate(ordered_labels , 0)}

    data[feature] = data[feature].map(labels_ordering)

data.head()

In [ ]:
from sklearn.model_selection import train_test_split
X = data.drop(['Id' , 'SalePrice'] , axis = 1)
y = data['SalePrice']
X_train , X_test , y_train , y_test = train_test_split(X,y,test_size = 0.2 , random_state = 42)


In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LassoCV
from sklearn.feature_selection import SelectFromModel

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

lasso_cv = LassoCV(cv = 5 , random_state = 42 , max_iter = 1000)

lasso_cv.fit(X_train_scaled , y_train)
print(f"🎯 The mathematically optimal alpha found by LassoCV is: {lasso_cv.alpha_:.6f}")

sel = SelectFromModel(lasso_cv , prefit = True )


selected_features = X_train.columns[sel.get_support()]

print(f"Total features: {X_train.shape[1]}")
print(f"Selected features: {len(selected_features)}")

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

X_train_final = X_train_scaled[:,sel.get_support()]
X_test_final = X_test_scaled[:,sel.get_support()]

linear_model = LinearRegression()
linear_model.fit(X_train_final , y_train) 

y_pred = linear_model.predict(X_test_final)

r2 = r2_score(y_test , y_pred)

print(f"🥇 Optimized Final Model R² Score: {r2:.4f}")

In [ ]:
import matplotlib.pyplot as plt

plt.scatter(y_test, y_pred, alpha=0.5, color='blue')
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
plt.xlabel('Actual SalePrice')
plt.ylabel('Predicted SalePrice')
plt.title('Actual vs. Predicted House Prices')
plt.show()